In [1]:
!pip install openrouter
!pip install rapidfuzz

In [27]:

os.chdir(r'C:\Users\SmartTech\Downloads')
print("Now working in:", os.getcwd())

Now working in: C:\Users\SmartTech\Downloads


In [41]:
import os
import json
from IPython.display import clear_output
os.environ["OPENROUTER_API_KEY"] = "notouchy!!!!!Awwwmad?"

In [5]:
API_KEY = os.environ.get("OPENROUTER_API_KEY")

In [31]:
import requests
import time

class ContentRefusalError(Exception):
    """Raised when the provider refuses the prompt for content-policy reasons."""
    pass

PROVIDER_MAP = {
    "openai/gpt-4o": "OpenAI",
    "openai/gpt-4o-mini": "OpenAI",
    "anthropic/claude-sonnet-4.5": "Amazon Bedrock",
    "meta-llama/llama-3.3-70b-instruct": "DeepInfra",
    "meta-llama/llama-3.1-8b-instruct": "DeepInfra"
}

def prompt(message, model, numtoken=100):

    for attempt in range(1, 11):

        try:
            request_json = {
                "model": model,
                "messages": [
                    {
                        "role": "user",
                        "content": message
                    }
                ],
                "max_tokens": numtoken,
                "temperature": 0
            }

            # =====================================================
            # PIN INFERENCE PROVIDER
            # =====================================================

            provider = PROVIDER_MAP.get(model)

            if provider is not None:
                request_json["provider"] = {
                    "only": [provider],
                    "allow_fallbacks": False
                }

            # =====================================================
            # API REQUEST
            # =====================================================

            response = requests.post(
                "https://openrouter.ai/api/v1/chat/completions",
                headers={
                    "Authorization": f"Bearer {API_KEY}",
                    "Content-Type": "application/json"
                },
                json=request_json,
                timeout=90
            )

            result = response.json()

            # =====================================================
            # API ERROR / PROVIDER ERROR
            # =====================================================

            if "choices" not in result:

                print(
                    f"API Error (attempt {attempt}) | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"response={result}"
                )

                time.sleep(2 * attempt)
                continue

            # =====================================================
            # GET RESPONSE
            # =====================================================

            choice = result["choices"][0]
            message_obj = choice["message"]

            content = (
                message_obj.get("content")
                or message_obj.get("reasoning")
                or ""
            )

            finish_reason = choice.get("finish_reason")
            native_finish_reason = choice.get("native_finish_reason")

            # =====================================================
            # CONTENT REFUSAL -- DO NOT RETRY
            # =====================================================

            if (
                finish_reason == "content_filter"
                or native_finish_reason == "refusal"
            ):

                print(
                    f"CONTENT REFUSAL -- skipping immediately | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"actual_provider={result.get('provider')} | "
                    f"finish_reason={finish_reason} | "
                    f"native_finish_reason={native_finish_reason}"
                )

                raise ContentRefusalError(
                    f"Content refusal | model={model} | provider={provider}"
                )

            # =====================================================
            # EMPTY RESPONSE -- RETRY IF NOT A REFUSAL
            # =====================================================

            if not content.strip():

                print(
                    f"Empty completion (attempt {attempt}) | "
                    f"model={model} | "
                    f"requested_provider={provider} | "
                    f"actual_provider={result.get('provider')} | "
                    f"finish_reason={finish_reason} | "
                    f"native_finish_reason={native_finish_reason}"
                )

                time.sleep(2 * attempt)
                continue

            # =====================================================
            # SUCCESS
            # =====================================================

            return content.strip()

        # =========================================================
        # IMPORTANT: REFUSALS ESCAPE RETRY LOOP
        # =========================================================

        except ContentRefusalError:
            raise

        except Exception as e:

            print(
                f"Exception (attempt {attempt}) | "
                f"model={model} | "
                f"provider={PROVIDER_MAP.get(model)} | "
                f"{e}"
            )

            time.sleep(2 * attempt)

    raise RuntimeError(
        f"API call failed after 10 attempts | "
        f"model={model} | "
        f"provider={PROVIDER_MAP.get(model)}"
    )

In [33]:
##These are functions assisting with relationship classification and graph construction, still part of premise construction phase
#A primary relationship classifier prompt, outputs causal evidence given eventTuple and question
class subGraph:
    def __init__(self, model):
        self.model = model
    def askCausalQ(self, context, graphType, cause, effect):
        if graphType == "conv":
            message = f"""
            You will be given an effect and a potential cause from a scenario.

            Scenario: {context}

            Effect: {effect}
            Potential Cause: {cause}

            If the potential cause mentioned above was removed or changed, would the effect change in any way?

            3: Likely yes, directly mentioned or implied
            2: Unsure, answer could be both yes or no
            1: Likely no, given the context of the question, the relationship requires too many assumptions
            0: The cause and effect refer to the same variable, or are too similar to be considered distinct variables

            Your output should be a single number with no explanation or any other redundant information.
            """
        elif graphType == "div":
            message = f"""
            You will be given an cause and a potential effect from a scenario.

            Scenario: {context}

            Cause: {cause}
            Potential Effect: {effect}

            If the cause mentioned above was removed or changed, would the potential effect change in any way?

            3: Likely yes, directly mentioned or implied
            2: Unsure, answer could be both yes or no
            1: Likely no, given the context of the question, the relationship requires too many assumptions
            0: Special Case - The cause and effect refer to the same variable, or are too similar to be considered distinct variables

            Your output should be a single number with no explanation or any other redundant information.
            """
        evidence = prompt(message, model=self.model, numtoken=10)
        return evidence.strip()

    def qualMap(self, num):
        if num==0:
          return 'unknown'
        if num==1:
            return 'Likely not a causal relationship'
        if num==2:
            return 'Ambiguous, answer could be yes or no.'
        if num==3:
            return 'Likely a causal relationship.'

    def constructGraph(self, entry):
      varList = entry['varList']
      target = entry['target']
      qType = entry['qType']
      context = entry['context']
      outputDict={}
      outputDict['Target'] = target
      outputDict['Question Type'] = qType
      edges = []
      if qType == "conv":
          for cause in varList:
              strength = self.askCausalQ(context, qType, cause, target)
              try:
                  strength = int(strength.strip()[0])
                  if strength == 0:
                    continue
              except:
                  strength = 1
              edges.append({'Cause': cause, 'Effect': target, 'Strength of Relationship': self.qualMap(strength)})
      elif qType == "div":
          for effect in varList:
              strength = self.askCausalQ(context, qType, target, effect)
              try:
                  strength = int(strength.strip()[0])
                  if strength == 0:
                    continue
              except:
                  strength = 1
              edges.append({'Cause': target, 'Effect': effect, 'Strength of Relationship': self.qualMap(strength)})
      outputDict['Edges'] = edges
      return outputDict

    def answer(self, graph, choices, context, question):
        strong_edges = [e for e in graph['Edges'] if e['Strength of Relationship'] == 'Likely a causal relationship.']
        if graph['Question Type'] == 'conv':
          qtypeText= 'This question asks for the CAUSE of the target variable'
        else:
          qtypeText = 'This question asks for the EFFECT of the target variable'
        filtered_graph = {**graph, 'Edges': strong_edges, 'Question Type': qtypeText}

        message = f"""
        You will be given a scenario, a question about a scenario, a causal graph visualizing the scenario, and answer choices. Your goal is to answer the question correctly.

        Context: {context}

        Question: {question}

        Causal Graph:
        {filtered_graph}

        Choice 1: {choices[0]}
        Choice 2: {choices[1]}
        Choice 3: {choices[2]}
        Choice 4: {choices[3]}

        Instructions:
        - Use the causal graph to help you understand which variables are most causally relevant to the question.
        - Also rely on the context and your understanding to select the best answer
        - The graph shows the target variable that the question about the context refers to, along with the question type (qType) telling what the question looks for.
        - The graph also includes a list of edges, representing causal relationships between each variable in the context and the target variable.
        - Use the information in the graph to help you better understand the problem.
        Output ONLY a single digit number (1, 2, 3, or 4). No explanation.
        """
        output = prompt(message, model=self.model, numtoken=10)
        try:
            output = int(output.strip())
        except:
            output = -1
        return output

    def runOnce(self, id, entry, debug=False, randomSelect = False):
        context = entry['context']
        question = entry['question']
        graph = self.constructGraph(entry)
        correctAnswer = entry['answer']
        choices = [entry['choice_1'], entry['choice_2'], entry['choice_3'], entry['choice_4']]
        selectedOption = self.answer(graph, choices, context, question)
        if debug:
            print(graph)
            print(selectedOption)
        if correctAnswer==selectedOption:
            return 1
        elif selectedOption == -1 and randomSelect:
            return 0
        elif selectedOption == -1 and not randomSelect:
            return f'id: {id} failed'
        else:
            return 0

def directLLM(entry, model):
  context = entry['context']
  question = entry['question']
  choices = [entry['choice_1'], entry['choice_2'], entry['choice_3'], entry['choice_4']]

  message = f"""
    You will be given a scenario, a question, and answer choices. Answer the question correctly.

    Context: {context}
    Question: {question}

    Choice 1: {choices[0]}
    Choice 2: {choices[1]}
    Choice 3: {choices[2]}
    Choice 4: {choices[3]}

    Output ONLY a single digit number (1, 2, 3, or 4). No explanation.
    """
  output = prompt(message, model, numtoken=10)
  try:
      return int(output.strip())
  except:
      return -1

In [45]:
models = [ 'anthropic/claude-sonnet-4.5',
          'meta-llama/llama-3.1-8b-instruct']

In [51]:
import time
import json
for model in models:
    print('running' + model)
    sg = subGraph(model)
    results = []
    baselineResults = []
    processedIds = []
    safe_name = model.replace('/', '_')
    for i, entry in enumerate(dataset):
        print(f"Processing {i+1}/3461 — ID is {entry['id']}")

        try:
            result = sg.runOnce(entry['id'], entry)
            results.append(result if isinstance(result, int) else 0)
        except Exception as e:
            print(f"  Subgraph error: {e}")
            results.append(0)

        try:
            predicted = directLLM(entry, model)
            baselineResults.append(1 if predicted == entry['answer'] else 0)
        except Exception as e:
            print(f"  Baseline error: {e}")
            baselineResults.append(0)

        processedIds.append(entry['id'])
        time.sleep(1)
        if (i+1) % 10 == 0:
            print(f"  Running... Subgraph: {sum(results)}/{len(results)}, Baseline: {sum(baselineResults)}/{len(baselineResults)}")
            with open(f'{safe_name}resultsCheckpoint.json', 'w') as f:
                json.dump({'results': results, 'baseline': baselineResults, 'ids': processedIds}, f)
            print(f"Answers Saved.")
        if (i+1) % 100 == 0:
            clear_output(wait=True)
    with open(f'{safe_name}resultsFinal.json', 'w') as f:
        json.dump({'results': results, 'baseline': baselineResults, 'ids': processedIds}, f)

    print(f"\nFinal {safe_name} Subgraph:   {sum(results)}/{len(results)} = {sum(results)/len(results):.2%}")
    print(f"Final {safe_name} Direct LLM: {sum(baselineResults)}/{len(baselineResults)} = {sum(baselineResults)/len(baselineResults):.2%}")

Processing 3101/3461 — ID is 3101
Processing 3102/3461 — ID is 3102
Processing 3103/3461 — ID is 3103
Processing 3104/3461 — ID is 3104
Processing 3105/3461 — ID is 3105
Processing 3106/3461 — ID is 3106
Processing 3107/3461 — ID is 3107
Processing 3108/3461 — ID is 3108
Processing 3109/3461 — ID is 3109
Processing 3110/3461 — ID is 3110
  Running... Subgraph: 1827/3110, Baseline: 2178/3110
Answers Saved.
Processing 3111/3461 — ID is 3111
Processing 3112/3461 — ID is 3112
Processing 3113/3461 — ID is 3113
Processing 3114/3461 — ID is 3114
Processing 3115/3461 — ID is 3115
Processing 3116/3461 — ID is 3116
Processing 3117/3461 — ID is 3117
Processing 3118/3461 — ID is 3118
Processing 3119/3461 — ID is 3119
Processing 3120/3461 — ID is 3120
  Running... Subgraph: 1834/3120, Baseline: 2186/3120
Answers Saved.
Processing 3121/3461 — ID is 3121
Processing 3122/3461 — ID is 3122
Processing 3123/3461 — ID is 3123
Processing 3124/3461 — ID is 3124
Processing 3125/3461 — ID is 3125
Processing 3

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



API Error (attempt 1) | model=meta-llama/llama-3.1-8b-instruct | requested_provider=DeepInfra | response={'error': {'message': 'A Timeout Occurred', 'code': 504}}
Processing 3418/3461 — ID is 3418
Processing 3419/3461 — ID is 3419
Processing 3420/3461 — ID is 3420
  Running... Subgraph: 2010/3420, Baseline: 2401/3420
Answers Saved.
Processing 3421/3461 — ID is 3421
Processing 3422/3461 — ID is 3422
Processing 3423/3461 — ID is 3423
Processing 3424/3461 — ID is 3424
Processing 3425/3461 — ID is 3425
Processing 3426/3461 — ID is 3426
Processing 3427/3461 — ID is 3427
Processing 3428/3461 — ID is 3428
Processing 3429/3461 — ID is 3429
Processing 3430/3461 — ID is 3430
  Running... Subgraph: 2016/3430, Baseline: 2411/3430
Answers Saved.
Processing 3431/3461 — ID is 3431
Processing 3432/3461 — ID is 3432
Processing 3433/3461 — ID is 3433
Processing 3434/3461 — ID is 3434
Processing 3435/3461 — ID is 3435
Processing 3436/3461 — ID is 3436
Processing 3437/3461 — ID is 3437
Processing 3438/346